In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import gymnasium as gym

# CRITICAL: Register Atari environments explicitly
try:
    # Try newer gymnasium approach
    import ale_py
    gym.register_envs(ale_py)
    print("✓ Registered Atari environments via ale_py")
except Exception as e:
    print(f"Note: {e}")
    # Fallback: try manual registration
    try:
        from ale_py import roms
        from ale_py.env import AtariEnv
        print("✓ ale_py imported, environments should be available")
    except Exception as e2:
        print(f"⚠ Could not import ale_py: {e2}")

import cv2
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

# Check gymnasium version and available environments
print(f"Gymnasium version: {gym.__version__}")

# Check for ALE
try:
    import ale_py
    print(f"ALE-py version: {ale_py.__version__}")
except:
    print("⚠ ALE-py not found")

print("\nSearching for Atari/Pong environments...")

# List all environments with 'Pong' in the name
pong_envs = [env_id for env_id in gym.envs.registry.keys() if 'pong' in env_id.lower()]
print(f"Found {len(pong_envs)} Pong-related environments:")
for env_id in sorted(pong_envs)[:10]:  # Show first 10
    print(f"  - {env_id}")

# Also check for ALE environments
ale_envs = [env_id for env_id in gym.envs.registry.keys() if 'ALE' in env_id]
print(f"\nFound {len(ale_envs)} ALE environments total")
if ale_envs:
    print("Sample ALE environments:", ale_envs[:5])

if not pong_envs:
    print("\n⚠ No Pong environments found!")
    print("Trying to create environment directly...")
    try:
        # Try direct environment creation with ALE
        test_env = gym.make('ALE/Pong-v5')
        print("✓ Successfully created ALE/Pong-v5 directly!")
        test_env.close()
        PONG_ENV_NAME = 'ALE/Pong-v5'
    except Exception as e:
        print(f"✗ Failed: {e}")
        PONG_ENV_NAME = None
else:
    PONG_ENV_NAME = pong_envs[0]
    print(f"\n✓ Will use: {PONG_ENV_NAME}")

✓ Registered Atari environments via ale_py

Using device: cuda
Gymnasium version: 1.2.2
ALE-py version: 0.11.2

Searching for Atari/Pong environments...
Found 5 Pong-related environments:
  - ALE/Pong-v5
  - Pong-v0
  - Pong-v4
  - PongNoFrameskip-v0
  - PongNoFrameskip-v4

Found 104 ALE environments total
Sample ALE environments: ['ALE/Adventure-v5', 'ALE/AirRaid-v5', 'ALE/Alien-v5', 'ALE/Amidar-v5', 'ALE/Assault-v5']

✓ Will use: Pong-v0


In [2]:
class AtariPreprocessing:
    """
    Standard Atari preprocessing for RL.
    - Converts 210×160×3 RGB frames → 84×84 grayscale
    - Stacks last 4 frames for temporal information
    - Frame skipping (takes action for N frames)
    """
    
    def __init__(self, frame_skip=4, frame_stack=4):
        self.frame_skip = frame_skip
        self.frame_stack = frame_stack
        self.frames = deque(maxlen=frame_stack)
        
    def reset(self):
        """Clear frame buffer."""
        self.frames.clear()
    
    def preprocess_frame(self, frame):
        """
        Convert 210×160×3 RGB → 84×84 grayscale.
        
        Args:
            frame: (210, 160, 3) RGB uint8 array
            
        Returns:
            (84, 84) grayscale float32 array, normalized to [0, 1]
        """
        # Convert to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        # Resize to 84×84
        resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
        
        # Normalize to [0, 1]
        normalized = resized.astype(np.float32) / 255.0
        
        return normalized
    
    def stack_frames(self, frame):
        """
        Stack last 4 frames for temporal information.
        
        Args:
            frame: (84, 84) preprocessed frame
            
        Returns:
            (4, 84, 84) stacked frames
        """
        self.frames.append(frame)
        
        # Pad if not enough frames yet (beginning of episode)
        while len(self.frames) < self.frame_stack:
            self.frames.append(frame)
        
        # Stack along channel dimension: (4, 84, 84)
        stacked = np.stack(list(self.frames), axis=0)
        return stacked
    
    def process(self, frame):
        """
        Full preprocessing pipeline: grayscale + resize + stack.
        
        Args:
            frame: (210, 160, 3) RGB frame from environment
            
        Returns:
            (4, 84, 84) stacked preprocessed frames
        """
        preprocessed = self.preprocess_frame(frame)
        stacked = self.stack_frames(preprocessed)
        return stacked

# Test preprocessing
print("Testing Atari preprocessing...")
env = gym.make('PongNoFrameskip-v4')
obs, _ = env.reset()

preprocessor = AtariPreprocessing()
processed = preprocessor.process(obs)

print(f"Original frame: {obs.shape} (210, 160, 3)")
print(f"Preprocessed: {processed.shape} (4, 84, 84)")
print(f"Range: [{processed.min():.3f}, {processed.max():.3f}]")
print("✓ Preprocessing works!")

env.close()

Testing Atari preprocessing...
Original frame: (210, 160, 3) (210, 160, 3)
Preprocessed: (4, 84, 84) (4, 84, 84)
Range: [0.251, 0.702]
✓ Preprocessing works!


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


In [3]:
class PongPolicyNetwork(nn.Module):
    """
    CNN policy network for Atari Pong.
    Architecture follows DQN/A3C standard:
    - 3 convolutional layers
    - 2 fully connected layers
    - Outputs action logits
    
    Input: (batch, 4, 84, 84) - 4 stacked grayscale frames
    Output: (batch, 3) - action logits for [NOOP, UP, DOWN]
    """
    
    def __init__(self, num_actions=3):
        super().__init__()
        
        # Convolutional layers (same as DQN)
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),   # (32, 20, 20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),  # (64, 9, 9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),  # (64, 7, 7)
            nn.ReLU()
        )
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, num_actions)
        )
        
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: (batch, 4, 84, 84) stacked frames
            
        Returns:
            logits: (batch, num_actions) raw action logits
        """
        # x should be (batch, 4, 84, 84)
        conv_out = self.conv(x)
        flat = conv_out.view(conv_out.size(0), -1)  # Flatten
        logits = self.fc(flat)
        return logits


class PongValueNetwork(nn.Module):
    """
    CNN value network for critic.
    Same architecture as policy network, but outputs single value.
    
    Input: (batch, 4, 84, 84) - 4 stacked grayscale frames
    Output: (batch, 1) - state value V(s)
    """
    
    def __init__(self):
        super().__init__()
        
        # Convolutional layers (same as policy)
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Single value output
        )
        
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: (batch, 4, 84, 84) stacked frames
            
        Returns:
            value: (batch, 1) state value
        """
        conv_out = self.conv(x)
        flat = conv_out.view(conv_out.size(0), -1)
        value = self.fc(flat)
        return value


# Test networks
print("Testing CNN networks...")

# Create dummy input (batch_size=2, 4 stacked frames of 84×84)
dummy_input = torch.randn(2, 4, 84, 84).to(device)

# Test policy network
policy = PongPolicyNetwork(num_actions=3).to(device)
logits = policy(dummy_input)
print(f"Policy network:")
print(f"  Input: {dummy_input.shape}")
print(f"  Output logits: {logits.shape}")
print(f"  Probabilities: {F.softmax(logits, dim=-1)[0]}")

# Test value network
value_net = PongValueNetwork().to(device)
values = value_net(dummy_input)
print(f"\nValue network:")
print(f"  Input: {dummy_input.shape}")
print(f"  Output values: {values.shape}")
print(f"  Sample values: {values.squeeze().detach().cpu().numpy()}")

# Count parameters
policy_params = sum(p.numel() for p in policy.parameters())
value_params = sum(p.numel() for p in value_net.parameters())
print(f"\nParameter counts:")
print(f"  Policy network: {policy_params:,} parameters")
print(f"  Value network: {value_params:,} parameters")

print("✓ Networks work!")

Testing CNN networks...
Policy network:
  Input: torch.Size([2, 4, 84, 84])
  Output logits: torch.Size([2, 3])
  Probabilities: tensor([0.3277, 0.3335, 0.3388], device='cuda:0', grad_fn=<SelectBackward0>)

Value network:
  Input: torch.Size([2, 4, 84, 84])
  Output values: torch.Size([2, 1])
  Sample values: [0.02243604 0.0150278 ]

Parameter counts:
  Policy network: 1,685,667 parameters
  Value network: 1,684,641 parameters
✓ Networks work!


# PPO Implementation for Pong

Now let's implement PPO adapted for visual inputs. This is similar to your drone PPO but handles image observations instead of state vectors.

In [4]:
def select_action(policy, state, temperature=1.0):
    """
    Select action using policy network with temperature-based sampling.
    
    Args:
        policy: Policy network
        state: (4, 84, 84) stacked frames
        temperature: Sampling temperature (0=greedy, 1=stochastic)
        
    Returns:
        action: Selected action (0, 1, or 2)
        log_prob: Log probability of action
    """
    with torch.no_grad():
        # Add batch dimension: (1, 4, 84, 84)
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        
        # Get logits
        logits = policy(state_tensor)
        
        # Apply temperature
        logits = logits / max(temperature, 1e-8)
        
        # Sample action
        probs = F.softmax(logits, dim=-1)
        action = torch.multinomial(probs, num_samples=1).item()
        
        # Get log probability
        log_prob = F.log_softmax(logits, dim=-1)[0, action].item()
        
    return action, log_prob


def collect_episode(env, policy, preprocessor, max_steps=10000):
    """
    Collect single episode using policy.
    
    Args:
        env: Gymnasium environment
        policy: Policy network
        preprocessor: AtariPreprocessing instance
        max_steps: Maximum episode length
        
    Returns:
        states: List of (4, 84, 84) stacked frames
        actions: List of actions
        rewards: List of rewards
        log_probs: List of log probabilities
        episode_reward: Total episode reward
    """
    states = []
    actions = []
    rewards = []
    log_probs = []
    
    # Reset environment and preprocessor
    obs, _ = env.reset()
    preprocessor.reset()
    state = preprocessor.process(obs)
    
    episode_reward = 0
    
    for step in range(max_steps):
        # Select action
        action, log_prob = select_action(policy, state, temperature=1.0)
        
        # Map action (we use 3 actions: 0=NOOP, 1=UP, 2=DOWN)
        # Pong action space: 0=NOOP, 2=UP, 3=DOWN
        pong_action = [0, 2, 3][action]
        
        # Execute action
        obs, reward, terminated, truncated, _ = env.step(pong_action)
        done = terminated or truncated
        
        # Store transition
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        log_probs.append(log_prob)
        
        episode_reward += reward
        
        if done:
            break
        
        # Process next state
        state = preprocessor.process(obs)
    
    return states, actions, rewards, log_probs, episode_reward


def compute_gae(rewards, values, next_value, gamma=0.99, gae_lambda=0.95):
    """
    Compute Generalized Advantage Estimation (GAE).
    
    Args:
        rewards: List of rewards
        values: List of state values V(s)
        next_value: Value of final state (0 if terminal)
        gamma: Discount factor
        gae_lambda: GAE lambda parameter
        
    Returns:
        advantages: List of advantages
        returns: List of returns (targets for value function)
    """
    advantages = []
    gae = 0
    
    # Compute GAE backwards
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_val = next_value
        else:
            next_val = values[t + 1]
        
        # TD error: δ = r + γ*V(s') - V(s)
        delta = rewards[t] + gamma * next_val - values[t]
        
        # GAE: A = δ + γ*λ*δ_{t+1} + (γ*λ)^2*δ_{t+2} + ...
        gae = delta + gamma * gae_lambda * gae
        advantages.insert(0, gae)
    
    # Returns: R = A + V
    returns = [adv + val for adv, val in zip(advantages, values)]
    
    return advantages, returns


In [5]:
def ppo_update(policy, value_net, policy_optimizer, value_optimizer,
               states, actions, old_log_probs, advantages, returns,
               clip_epsilon=0.2, value_loss_coef=0.5, entropy_coef=0.01,
               num_epochs=4, batch_size=64, max_grad_norm=0.5):
    """
    PPO update with clipped objective.
    
    Args:
        policy: Policy network
        value_net: Value network
        policy_optimizer: Optimizer for policy
        value_optimizer: Optimizer for value network
        states: List of states
        actions: List of actions
        old_log_probs: List of old log probabilities
        advantages: List of advantages
        returns: List of returns (value targets)
        clip_epsilon: PPO clip parameter
        value_loss_coef: Value loss coefficient
        entropy_coef: Entropy bonus coefficient
        num_epochs: Number of PPO epochs
        batch_size: Batch size for updates
        max_grad_norm: Max gradient norm for clipping
        
    Returns:
        policy_loss: Average policy loss
        value_loss: Average value loss
        entropy: Average entropy
    """
    # Convert to tensors
    states_tensor = torch.tensor(np.array(states), dtype=torch.float32).to(device)
    actions_tensor = torch.tensor(actions, dtype=torch.long).to(device)
    old_log_probs_tensor = torch.tensor(old_log_probs, dtype=torch.float32).to(device)
    advantages_tensor = torch.tensor(advantages, dtype=torch.float32).to(device)
    returns_tensor = torch.tensor(returns, dtype=torch.float32).to(device)
    
    # Normalize advantages
    advantages_tensor = (advantages_tensor - advantages_tensor.mean()) / (advantages_tensor.std() + 1e-8)
    
    total_policy_loss = 0
    total_value_loss = 0
    total_entropy = 0
    num_updates = 0
    
    # PPO epochs
    for epoch in range(num_epochs):
        # Shuffle data
        indices = np.random.permutation(len(states))
        
        # Mini-batch updates
        for start_idx in range(0, len(states), batch_size):
            end_idx = min(start_idx + batch_size, len(states))
            batch_indices = indices[start_idx:end_idx]
            
            # Get batch
            batch_states = states_tensor[batch_indices]
            batch_actions = actions_tensor[batch_indices]
            batch_old_log_probs = old_log_probs_tensor[batch_indices]
            batch_advantages = advantages_tensor[batch_indices]
            batch_returns = returns_tensor[batch_indices]
            
            # === Policy update ===
            logits = policy(batch_states)
            log_probs = F.log_softmax(logits, dim=-1)
            
            # Get log prob of taken actions
            action_log_probs = log_probs.gather(1, batch_actions.unsqueeze(1)).squeeze()
            
            # Compute ratio: π(a|s) / π_old(a|s)
            ratio = torch.exp(action_log_probs - batch_old_log_probs)
            
            # Clipped surrogate objective
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Entropy bonus (encourage exploration)
            probs = F.softmax(logits, dim=-1)
            entropy = -(probs * log_probs).sum(dim=-1).mean()
            
            # Total policy loss
            total_policy_objective = policy_loss - entropy_coef * entropy
            
            # Update policy
            policy_optimizer.zero_grad()
            total_policy_objective.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
            policy_optimizer.step()
            
            # === Value update ===
            values = value_net(batch_states).squeeze()
            value_loss = F.mse_loss(values, batch_returns)
            
            # Update value network
            value_optimizer.zero_grad()
            value_loss.backward()
            torch.nn.utils.clip_grad_norm_(value_net.parameters(), max_grad_norm)
            value_optimizer.step()
            
            # Track losses
            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
            total_entropy += entropy.item()
            num_updates += 1
    
    avg_policy_loss = total_policy_loss / num_updates
    avg_value_loss = total_value_loss / num_updates
    avg_entropy = total_entropy / num_updates
    
    return avg_policy_loss, avg_value_loss, avg_entropy


print("✓ PPO update function defined!")
print("  - Clipped surrogate objective")
print("  - Entropy bonus for exploration")
print("  - Gradient clipping for stability")
print("  - Multi-epoch mini-batch training")

✓ PPO update function defined!
  - Clipped surrogate objective
  - Entropy bonus for exploration
  - Gradient clipping for stability
  - Multi-epoch mini-batch training


# Training Setup

Let's initialize the networks and training configuration.

In [6]:
# Hyperparameters (from your self-play plan + drone experience)
LEARNING_RATE = 2.5e-4
GAMMA = 0.99              # Discount factor
GAE_LAMBDA = 0.95         # GAE lambda
CLIP_EPSILON = 0.2        # PPO clip parameter
ENTROPY_COEF = 0.01       # Entropy bonus
VALUE_LOSS_COEF = 0.5     # Value loss coefficient
MAX_GRAD_NORM = 0.5       # Gradient clipping
NUM_EPOCHS = 4            # PPO epochs per update
BATCH_SIZE = 64           # Mini-batch size
NUM_EPISODES_PER_ITER = 5 # Episodes to collect per iteration

# Initialize networks
policy = PongPolicyNetwork(num_actions=3).to(device)
value_net = PongValueNetwork().to(device)

# Optimizers
policy_optimizer = optim.Adam(policy.parameters(), lr=LEARNING_RATE)
value_optimizer = optim.Adam(value_net.parameters(), lr=LEARNING_RATE)

# Preprocessing
preprocessor = AtariPreprocessing()

# Environment
env = gym.make('PongNoFrameskip-v4')

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"Device: {device}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Gamma: {GAMMA}")
print(f"GAE lambda: {GAE_LAMBDA}")
print(f"Clip epsilon: {CLIP_EPSILON}")
print(f"Entropy coefficient: {ENTROPY_COEF}")
print(f"Batch size: {BATCH_SIZE}")
print(f"PPO epochs: {NUM_EPOCHS}")
print(f"Episodes per iteration: {NUM_EPISODES_PER_ITER}")
print("=" * 60)
print(f"Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")
print(f"Value parameters: {sum(p.numel() for p in value_net.parameters()):,}")
print("=" * 60)

TRAINING CONFIGURATION
Device: cuda
Learning rate: 0.00025
Gamma: 0.99
GAE lambda: 0.95
Clip epsilon: 0.2
Entropy coefficient: 0.01
Batch size: 64
PPO epochs: 4
Episodes per iteration: 5
Policy parameters: 1,685,667
Value parameters: 1,684,641


# Training Loop

Let's test the training loop with a few iterations to make sure everything works before committing to longer training.

In [7]:
def train_iteration(policy, value_net, policy_optimizer, value_optimizer, 
                    env, preprocessor, num_episodes=5):
    """
    Single training iteration: collect episodes + PPO update.
    
    Returns:
        metrics: Dict with training metrics
    """
    all_states = []
    all_actions = []
    all_old_log_probs = []
    all_advantages = []
    all_returns = []
    episode_rewards = []
    
    policy.eval()
    value_net.eval()
    
    # Collect episodes
    for ep in range(num_episodes):
        states, actions, rewards, log_probs, episode_reward = collect_episode(
            env, policy, preprocessor, max_steps=10000
        )
        
        episode_rewards.append(episode_reward)
        
        # Compute values for GAE
        with torch.no_grad():
            states_tensor = torch.tensor(np.array(states), dtype=torch.float32).to(device)
            values = value_net(states_tensor).squeeze().cpu().numpy()
        
        # Compute advantages using GAE
        advantages, returns = compute_gae(
            rewards, values.tolist(), next_value=0.0,
            gamma=GAMMA, gae_lambda=GAE_LAMBDA
        )
        
        # Store episode data
        all_states.extend(states)
        all_actions.extend(actions)
        all_old_log_probs.extend(log_probs)
        all_advantages.extend(advantages)
        all_returns.extend(returns)
    
    # PPO update
    policy.train()
    value_net.train()
    
    policy_loss, value_loss, entropy = ppo_update(
        policy, value_net, policy_optimizer, value_optimizer,
        all_states, all_actions, all_old_log_probs, all_advantages, all_returns,
        clip_epsilon=CLIP_EPSILON,
        value_loss_coef=VALUE_LOSS_COEF,
        entropy_coef=ENTROPY_COEF,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        max_grad_norm=MAX_GRAD_NORM
    )
    
    metrics = {
        'avg_episode_reward': np.mean(episode_rewards),
        'min_episode_reward': np.min(episode_rewards),
        'max_episode_reward': np.max(episode_rewards),
        'policy_loss': policy_loss,
        'value_loss': value_loss,
        'entropy': entropy,
        'num_steps': len(all_states)
    }
    
    return metrics


# Test training loop (just 2 iterations to verify it works)
print("Testing training loop...")
print("This will take a few minutes...\n")

history = []

for iteration in range(2):
    metrics = train_iteration(
        policy, value_net, policy_optimizer, value_optimizer,
        env, preprocessor, num_episodes=NUM_EPISODES_PER_ITER
    )
    
    history.append(metrics)
    
    print(f"Iteration {iteration + 1}")
    print(f"  Avg reward: {metrics['avg_episode_reward']:.2f}")
    print(f"  Min/Max: [{metrics['min_episode_reward']:.1f}, {metrics['max_episode_reward']:.1f}]")
    print(f"  Steps: {metrics['num_steps']}")
    print(f"  Policy loss: {metrics['policy_loss']:.4f}")
    print(f"  Value loss: {metrics['value_loss']:.4f}")
    print(f"  Entropy: {metrics['entropy']:.4f}")
    print()

print("✓ Training loop works!")

Testing training loop...
This will take a few minutes...

Iteration 1
  Avg reward: -21.00
  Min/Max: [-21.0, -21.0]
  Steps: 18668
  Policy loss: -0.0019
  Value loss: 0.0259
  Entropy: 1.0909

Iteration 2
  Avg reward: -21.00
  Min/Max: [-21.0, -21.0]
  Steps: 16972
  Policy loss: -0.0006
  Value loss: 0.0145
  Entropy: 1.0778

✓ Training loop works!


# Self-Play: Playing Against Yourself

## The Problem with Built-in AI

Pong has a built-in AI opponent, but it has **fixed difficulty**. Your agent might:
1. Learn to exploit specific weaknesses in the built-in AI
2. Overfit to that one opponent
3. Not generalize well

## Self-Play Solution

Instead of playing against the built-in AI, we make **two agents**:
- **Player 1 (Learning Agent)**: The agent we're training (gets updated)
- **Player 2 (Opponent)**: A **frozen copy** of the agent from earlier iterations

### Key Idea:
- Every N iterations, we **freeze** the current agent and use it as the opponent
- The learning agent now plays against its past self
- As the agent improves, opponents automatically get harder!

### Implementation Strategy:

```python
# Iteration 0-50: Agent vs random/weak opponent
# Iteration 50: Freeze agent → becomes "Opponent v1"
# Iteration 51-100: Agent vs Opponent v1
# Iteration 100: Freeze agent → becomes "Opponent v2"
# Iteration 101-150: Agent vs Opponent v2
# ...and so on
```

### Population-Based Training (Advanced):

Instead of always playing against the most recent opponent, maintain a **pool of past opponents**:
```python
opponent_pool = [agent_v1, agent_v2, agent_v3, ..., agent_v10]

# Each training iteration:
opponent = random.choice(opponent_pool)  # Sample random past version
agent.play_against(opponent)
agent.update()

# Every 50 iterations:
opponent_pool.append(agent.copy())  # Add current agent to pool
if len(opponent_pool) > 10:
    opponent_pool.pop(0)  # Keep last 10 versions
```

## Two Approaches for Pong:

### Approach 1: Two-Player Pong (Harder to implement)
- Modify Pong to have two controllable paddles
- Both agents control their own paddle
- Requires custom environment

### Approach 2: Curriculum Learning (Easier to start)
- Train against built-in AI first (baseline)
- Then implement self-play once agent is competent
- OR: Use the built-in AI difficulty levels as curriculum

**For now, let's start with Approach 2** (train against built-in AI to get baseline working), then we can add self-play later.

In [ ]:
import copy

class SelfPlayPong:
    """
    Self-play trainer for Pong.
    
    The challenge: Pong is a two-player game, but gymnasium's Pong
    only lets you control ONE paddle (the other is built-in AI).
    
    Solution: We need to use a two-player Pong environment OR
    implement our own wrapper that allows both agents to play.
    """
    
    def __init__(self, learning_agent, opponent=None):
        self.agent = learning_agent  # The agent being trained
        self.opponent = opponent if opponent else copy.deepcopy(learning_agent)
        self.opponent.eval()  # Freeze opponent (no training)
        
        # Opponent pool for population-based training
        self.opponent_pool = []
        
    def update_opponent(self, strategy='latest'):
        """
        Update opponent to a frozen copy of current agent.
        
        Args:
            strategy: 'latest' or 'population'
        """
        if strategy == 'latest':
            # Freeze current agent as new opponent
            self.opponent = copy.deepcopy(self.agent)
            self.opponent.eval()
            print("✓ Opponent updated to latest agent version")
            
        elif strategy == 'population':
            # Add to pool and sample random opponent
            self.opponent_pool.append(copy.deepcopy(self.agent))
            if len(self.opponent_pool) > 10:
                self.opponent_pool.pop(0)  # Keep last 10
            
            # Sample random opponent from pool
            self.opponent = copy.deepcopy(np.random.choice(self.opponent_pool))
            self.opponent.eval()
            print(f"✓ Opponent sampled from pool (size: {len(self.opponent_pool)})")
    
    def collect_self_play_episode(self, env, preprocessor):
        """
        Collect episode where agent plays against opponent.
        
        PROBLEM: Standard Pong environment doesn't support two controllable agents!
        
        WORKAROUND: We'll start with single-player Pong (vs built-in AI),
        then later we can:
        1. Use PettingZoo's two-player Pong environment
        2. Build our own two-player Pong wrapper
        3. Use a different game (Connect Four is easier for self-play)
        """
        # For now, just use regular Pong vs built-in AI
        states, actions, rewards, log_probs, episode_reward = collect_episode(
            env, self.agent, preprocessor
        )
        
        return states, actions, rewards, log_probs, episode_reward


# Initialize self-play trainer
print("Self-play trainer initialized")
print("\n⚠ IMPORTANT NOTE:")
print("Standard gymnasium Pong only allows ONE player (vs built-in AI).")
print("For true self-play, we need:")
print("  1. PettingZoo's pong_v3 (two-player version)")
print("  2. OR start with Connect Four (easier for self-play)")
print("  3. OR build custom two-player Pong environment")
print("\nFor now, we'll train vs built-in AI as baseline.")

# Installing PettingZoo for True Self-Play Pong

PettingZoo provides multi-agent environments where **both players are controllable**.

Let's install it and set up two-player Pong.

In [ ]:
# Install PettingZoo (run this once)
# Uncomment and run if not installed:
# !pip install pettingzoo[atari]

# Import PettingZoo
try:
    from pettingzoo.atari import pong_v3
    print("✓ PettingZoo imported successfully")
    
    # Test environment creation
    env = pong_v3.env()
    print(f"✓ Created pong_v3 environment")
    print(f"  Agents: {env.agents}")
    print(f"  Observation space: {env.observation_space('first_0')}")
    print(f"  Action space: {env.action_space('first_0')}")
    env.close()
    
except ImportError as e:
    print(f"⚠ PettingZoo not installed: {e}")
    print("\nInstall with:")
    print("  pip install 'pettingzoo[atari]'")
except Exception as e:
    print(f"⚠ Error: {e}")
    print("\nYou may also need:")
    print("  pip install gymnasium[accept-rom-license]  # For Atari ROMs")